# Molecular Property Prediction with Chemprop
## AI for Science Demo — Drug Discovery Workshop

**Core Question:** Given a molecule's SMILES structure, can we predict its aqueous solubility using a graph neural network?

**Approach:** Chemprop (MPNN) trained on the ESOL dataset → predict LogS (log solubility in mol/L)

## 1. Setup & Imports

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors

# Chemprop v2
from chemprop.data import MoleculeDatapoint, MoleculeDataset, build_dataloader
from chemprop.models import MPNN
from chemprop.nn import BondMessagePassing, MeanAggregation
from chemprop.data import StandardScaler

print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}  |  RDKit {Chem.rdkitVersion}")

## 2. Load Trained Model + Scaler

The scaler is **critical**: training targets were normalized (mean=0, std=1).
We must apply inverse transform to get predictions in the original LogS scale.

In [ ]:
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

# Find checkpoint
checkpoint_path = os.path.join(MODELS_DIR, "model_esol_train.pt")
if not os.path.exists(checkpoint_path):
    checkpoint_path = os.path.join(MODELS_DIR, "model_esol_train_100.pt")

results_path = os.path.join(MODELS_DIR, "results_esol_train.json")
if not os.path.exists(results_path):
    results_path = os.path.join(RESULTS_DIR, "results_scale100_rep0.json")

print(f"Checkpoint: {checkpoint_path}")
print(f"Results:    {results_path}")

# Load checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device)

# ── Read architecture from checkpoint args (robust, not hardcoded) ──
if "args" in checkpoint:
    ckpt_args = checkpoint["args"]
    hidden = ckpt_args.get("hidden_size", 300)
    depth  = ckpt_args.get("depth", 5)
    dropout = ckpt_args.get("dropout", 0.1)
    print(f"Arch from checkpoint: hidden={hidden}, depth={depth}, dropout={dropout}")
else:
    hidden, depth, dropout = 300, 5, 0.1  # fallback
    print(f"Arch from fallback:  hidden={hidden}, depth={depth}, dropout={dropout}")

# Rebuild model
mp_block = BondMessagePassing(d_v=hidden, d_e=14, depth=depth)
agg = MeanAggregation()
model = MPNN(
    message_passing=mp_block, agg=agg,
    batch_norm=True, dropout=dropout,
)

if "model_state" in checkpoint:
    model.load_state_dict(checkpoint["model_state"])
else:
    model.load_state_dict(checkpoint)
model.to(device)
model.eval()

# ── Build scaler from training data (for inverse normalization) ──
# We re-fit the scaler on the training set so we can invert predictions.
train_csv = os.path.join(DATA_DIR, "esol_train.csv")
if os.path.exists(train_csv):
    train_df = pd.read_csv(train_csv)
    train_dps = [MoleculeDatapoint.from_smi(r["smiles"], targets=[r["logS"]]) for _, r in train_df.iterrows()]
    train_ds = MoleculeDataset(train_dps)
    scaler = StandardScaler()
    scaler.fit(train_ds)
    print(f"Scaler fitted: mean={scaler.mean_[0]:.3f}, std={scaler.scale_[0]:.3f}")
else:
    scaler = None
    print("WARNING: Training CSV not found — predictions will be in normalized space")

# Load metrics
with open(results_path) as f:
    results = json.load(f)
print(f"\nModel Performance on Test Set:")
print(f"  RMSE = {results['rmse']:.3f} LogS")
print(f"  MAE  = {results['mae']:.3f} LogS")
print(f"  R²   = {results['r2']:.3f}")
print(f"  Trained on {results.get('n_train', '?')} molecules")
print("\nReady for inference.")

## 3. Inference Function (with inverse normalization)

In [ ]:
def predict_logS(smiles: str, show_structure: bool = True):
    """
    Predict aqueous solubility (LogS) for a given SMILES string.
    Applies inverse normalization to return real LogS values.
    """
    # Validate SMILES
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    # Create datapoint and predict
    datapoint = MoleculeDatapoint.from_smi(smiles)
    dataset = MoleculeDataset([datapoint])
    loader = build_dataloader(dataset, batch_size=1, shuffle=False)

    with torch.no_grad():
        for batch in loader:
            bmg, V_d, *_ = batch
            bmg = bmg.to(device)
            V_d = V_d.to(device)
            pred = model(bmg, V_d)
            pred_norm = pred.cpu().numpy().reshape(-1, 1)

    # ── CRITICAL: Inverse normalization to get real LogS ──
    if scaler is not None:
        pred_real = scaler.inverse_transform(pred_norm).flatten()
        logS = float(pred_real[0])
    else:
        logS = float(pred_norm[0, 0])

    # Display structure
    if show_structure:
        img = Draw.MolToImage(mol, size=(300, 200))
        display(img)

    # Interpret solubility
    if logS > 0:
        level = "High — very soluble in water"
        meaning = "Dissolves readily. Good for oral drug formulation."
    elif logS > -2:
        level = "Moderate — reasonably soluble"
        meaning = "Acceptable for most drug development purposes."
    elif logS > -4:
        level = "Low — poorly soluble"
        meaning = "May need formulation strategies (salt forms, nanoparticles)."
    else:
        level = "Very low — essentially insoluble"
        meaning = "Major challenge for oral bioavailability."

    print(f"SMILES:          {smiles}")
    print(f"Predicted LogS:  {logS:.2f} mol/L")
    print(f"Solubility:      {level}")
    print(f"  → {meaning}")

    # Additional properties from RDKit (no ML needed)
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Descriptors.NumHDonors(mol)
    hba = Descriptors.NumHAcceptors(mol)
    print(f"\nMolecular Properties (RDKit):")
    print(f"  MW: {mw:.1f} g/mol  |  LogP: {logp:.2f}  |  HBD: {hbd}  |  HBA: {hba}")

    return logS

## 4. Live Demo — Predict Solubility

Enter any SMILES string below to predict its aqueous solubility.

In [ ]:
# ── DEMO: Try your own molecule! ──
# Replace the SMILES below with any valid SMILES string

my_smiles = "CC(=O)Oc1ccccc1C(=O)O"  # Aspirin

predicted_logS = predict_logS(my_smiles)

### Try These Well-Known Molecules

In [ ]:
demo_molecules = {
    "Aspirin":      "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":     "CN1C=NC2=C1C(=O)N(C(=O)N2C)C",
    "Benzene":      "c1ccccc1",
    "Ibuprofen":    "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "Paracetamol":  "CC(=O)NC1=CC=C(C=C1)O",
    "Glucose":      "C(C1C(C(C(C(O1)O)O)O)O)O",
    "Testosterone": "CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C",
    "Vitamin C":    "C(C(C1C(=C(C(=O)O1)O)O)O)O",
}

predictions = []
for name, smi in demo_molecules.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    try:
        logS = predict_logS(smi, show_structure=False)
        predictions.append({"Molecule": name, "Predicted LogS": f"{logS:.2f}"})
    except Exception as e:
        print(f"  Error: {e}")

if predictions:
    print(f"\n{'='*50}")
    print("  Summary")
    print(f"{'='*50}")
    df = pd.DataFrame(predictions)
    print(df.to_string(index=False))

## 5. Results Overview (Pre-generated Figures)

In [ ]:
from IPython.display import Image, display

fig_dir = RESULTS_DIR

print("Figure 1: Predicted vs True Solubility")
fig1 = os.path.join(fig_dir, "predicted_vs_true.png")
if os.path.exists(fig1):
    display(Image(fig1, width=500))
else:
    print(f"  (Run scripts/train.py then scripts/visualize.py first)")

print("\nFigure 2: Error Distribution")
fig2 = os.path.join(fig_dir, "error_distribution.png")
if os.path.exists(fig2):
    display(Image(fig2, width=550))

print("\nFigure 3: Effect of Training Data Size")
fig3 = os.path.join(fig_dir, "data_scale_rmse.png")
if os.path.exists(fig3):
    display(Image(fig3, width=700))

print("\nFigure 4: Training Curves")
fig4 = os.path.join(fig_dir, "training_curves.png")
if os.path.exists(fig4):
    display(Image(fig4, width=500))

## 6. Key Takeaways

1. **Graph Neural Networks** treat molecules as graphs (atoms = nodes, bonds = edges), naturally capturing molecular topology without hand-crafted features.

2. **Data quantity matters:** Our data-scale experiment shows RMSE improves substantially as training data grows — validating the 'AI for Science' principle that data defines the upper bound of what models can learn.

3. **Solubility is a practical bottleneck:** If a drug candidate doesn't dissolve in water, it cannot be absorbed. AI screening can filter out insoluble candidates before expensive synthesis.

4. **Chemprop lowers the barrier:** Modular, well-documented, PyTorch-based. A few hundred lines of Python is enough for a complete molecular ML pipeline.

---
*References: Delaney, J.S. J. Chem. Inf. Comput. Sci. 2004, 44, 1000–1005 | Chemprop v2: Heid et al. J. Chem. Inf. Model. 2025*